# Quant Skills — Lesson 8: Rolling Hedge Ratios and a Basket of Pairs

Lesson 7 found the problem: **one pair, with a relationship fixed once, breaks.** This lesson fixes both halves
the way stat-arb desks do:

1. **A rolling hedge ratio.** A **Kalman filter** re-estimates the relationship between the two stocks every single day,
   using only information available up to that day. When the relationship drifts, the hedge drifts with it.
2. **A basket.** Instead of betting on one pair, trade **15 pairs at once** — each with a real business reason to be linked.
   No single breakdown sinks the portfolio.

**How to run it:** **Runtime → Run all** (one to two minutes). Lines you can edit are marked `# <- EDIT`.

## Step 1 — Load the tools

In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
print("Tools loaded.")

## Step 2 — The basket

Every pair here is chosen for a **business reason** — same industry, same customers, same economic drivers — *before*
looking at any results. That avoids the multiple-testing trap from Lesson 7: we're not searching for pairs that
happened to work, we're trading all of them.

In [ ]:
PAIRS = [   # <- EDIT: add or remove (stock A, stock B) pairs from the same industry
    ("V", "MA"),     ("HD", "LOW"),   ("KO", "PEP"),   ("XOM", "CVX"),  ("JPM", "BAC"),
    ("GS", "MS"),    ("UPS", "FDX"),  ("T", "VZ"),     ("PG", "CL"),    ("LMT", "NOC"),
    ("MRK", "PFE"),  ("DAL", "UAL"),  ("CAT", "DE"),   ("WMT", "COST"), ("MCD", "YUM"),
]
YEARS = 5      # <- EDIT
TRAIN = 0.6    # the static comparison is fit on the first 60%; everything is judged on the last 40%

tickers = sorted({t for p in PAIRS for t in p} | {"SPY"})
raw = yf.download(tickers, period=f"{YEARS}y", interval="1d", auto_adjust=True, progress=False)
close = raw["Close"][tickers].dropna(how="all").ffill().dropna()
split = close.index[int(len(close) * TRAIN)]
PAIRS = [p for p in PAIRS if p[0] in close and p[1] in close]
print(f"{len(PAIRS)} pairs · {len(close)} trading days · judged on the period after {split.date()}")

## Step 3 — The Kalman filter: a hedge ratio that updates every day

Each morning the filter asks: *given everything up to yesterday, what should the relationship between these two
stocks be today?* It predicts today's price of A from today's price of B, measures the **surprise** (the forecast
error), then nudges its estimate of the relationship toward what it just saw.

- A **big surprise relative to normal** → the pair is stretched → trade it.
- The **hedge ratio drifts** as the relationship changes, instead of staying frozen like in Lesson 7.

`DELTA` controls how fast it adapts. This is the key setting:
- **Too fast** and the filter absorbs the very gap you're trying to trade — every stretch looks "normal" by the next day.
- **Too slow** and it never learns the relationship at all.

In [ ]:
DELTA = 1e-7   # <- EDIT: how quickly the hedge ratio adapts (try 1e-6 for faster, 1e-8 for slower)
VE    = 1e-3   # how noisy we assume each day's observation is

def kalman_pair(y, x, delta=DELTA, ve=VE):
    """Returns the daily hedge ratio (beta), the forecast surprise, and its expected size."""
    n = len(y)
    Vw = delta / (1 - delta) * np.eye(2)
    theta, P = np.zeros(2), np.zeros((2, 2))
    beta, err, q = np.zeros(n), np.zeros(n), np.zeros(n)
    for t in range(n):
        F = np.array([x[t], 1.0])
        R = P + Vw if t > 0 else np.zeros((2, 2))
        e = y[t] - F @ theta                 # surprise, using only yesterday's estimate
        Q = F @ R @ F + ve
        K = R @ F / Q
        theta = theta + K * e
        P = R - np.outer(K, F @ R)
        beta[t], err[t], q[t] = theta[0], e, Q
    return beta, err, q

a, b = PAIRS[0]
beta_ex, _, _ = kalman_pair(np.log(close[a].values), np.log(close[b].values))
plt.figure(figsize=(10, 3.8))
plt.plot(close.index[60:], beta_ex[60:], linewidth=1.5)
plt.axvline(split, color="red", linestyle="--", linewidth=1, label="judging period starts")
plt.title(f"{a}/{b}: the hedge ratio, re-estimated every day"); plt.ylabel("Hedge ratio"); plt.legend(); plt.grid(alpha=0.3)
plt.show()
print("A flat line would mean a stable relationship. Drift is exactly what broke the fixed hedge in Lesson 7.")

## Step 4 — Trading rules for every pair

| Signal | Action |
|---|---|
| Surprise above **+1.5** of normal (measured against the last 60 days of surprises) | Spread too wide → short A, buy B |
| Surprise below **−1.5** of normal | Spread too narrow → buy A, short B |
| Back inside **±0.25** | Exit |
| Beyond **±4**, or held **30 days** | Stop out |

Costs are charged on every entry and exit **and** on the daily hedge adjustments.

In [ ]:
ENTRY_Z, EXIT_Z, STOP_Z, MAX_HOLD = 1.5, 0.25, 4.0, 30   # <- EDIT
COST   = 0.001   # 0.10% per side per leg — commissions, slippage, short borrow
WARMUP = 60      # give the filter time to settle before trading

def trade_signals(z):
    pos, held, out = 0, 0, np.zeros(len(z))
    for i, zi in enumerate(z):
        if i < WARMUP or np.isnan(zi):
            pos = 0
        elif pos == 0:
            if zi > ENTRY_Z: pos, held = -1, 0
            elif zi < -ENTRY_Z: pos, held = 1, 0
        else:
            held += 1
            if abs(zi) < EXIT_Z or abs(zi) > STOP_Z or held > MAX_HOLD:
                pos = 0
        out[i] = pos
    return out

def pair_returns_kalman(a, b):
    y, x = np.log(close[a].values), np.log(close[b].values)
    beta, err, q = kalman_pair(y, x)
    err_s = pd.Series(err)
    z = (err_s / err_s.rolling(60).std().shift(1)).values   # "normal" = the last 60 days of surprises, known before today
    pos = trade_signals(z)
    ra, rb = close[a].pct_change().values, close[b].pct_change().values
    gross = 1 + np.abs(beta)
    r = np.zeros(len(y))
    r[1:] = pos[:-1] * (ra[1:] - beta[:-1] * rb[1:]) / gross[:-1]
    trade_cost = np.abs(np.diff(pos, prepend=0)) * COST
    hedge_cost = np.abs(pos) * np.abs(np.diff(beta, prepend=beta[0])) / gross * COST
    return pd.Series(r - trade_cost - hedge_cost, index=close.index), pos

def pair_returns_static(a, b):
    tr = close[close.index < split]
    beta, alpha = np.polyfit(np.log(tr[b]), np.log(tr[a]), 1)
    spread = np.log(close[a]) - beta * np.log(close[b])
    z = ((spread - spread.rolling(60).mean()) / spread.rolling(60).std()).values
    pos = trade_signals(z)
    ra, rb = close[a].pct_change().values, close[b].pct_change().values
    r = np.zeros(len(z))
    r[1:] = pos[:-1] * (ra[1:] - beta * rb[1:]) / (1 + abs(beta))
    r -= np.abs(np.diff(pos, prepend=0)) * COST
    return pd.Series(r, index=close.index), pos

kal, sta, n_trades = {}, {}, {}
for a, b in PAIRS:
    kal[f"{a}/{b}"], pos = pair_returns_kalman(a, b)
    sta[f"{a}/{b}"], _ = pair_returns_static(a, b)
    test_pos = pos[close.index >= split]
    n_trades[f"{a}/{b}"] = int((np.diff(test_pos, prepend=0) != 0).sum() // 2)
kal, sta = pd.DataFrame(kal), pd.DataFrame(sta)
print("Backtests complete.")

## Step 5 — Pair by pair, on the judging period

In [ ]:
def sharpe(r):
    return r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else np.nan

test = close.index >= split
rows = []
for name in kal.columns:
    rk, rs = kal.loc[test, name], sta.loc[test, name]
    rows.append({"Pair": name, "Rolling: return": (1 + rk).prod() - 1, "Rolling: Sharpe": sharpe(rk),
                 "Fixed: return": (1 + rs).prod() - 1, "Fixed: Sharpe": sharpe(rs), "Trades": n_trades[name]})
tbl = pd.DataFrame(rows).sort_values("Rolling: Sharpe", ascending=False)
show = tbl.copy()
for c in ("Rolling: return", "Fixed: return"): show[c] = show[c].map("{:+.1%}".format)
for c in ("Rolling: Sharpe", "Fixed: Sharpe"): show[c] = show[c].map("{:+.2f}".format)
print(show.to_string(index=False))
print()
print(f"Pairs profitable — rolling hedge: {(tbl['Rolling: return'] > 0).sum()} of {len(tbl)}   ·   "
      f"fixed hedge: {(tbl['Fixed: return'] > 0).sum()} of {len(tbl)}")

## Step 6 — The portfolio: all pairs together

Capital is split evenly across every pair. On days a pair has no trade, its share of capital earns nothing.
The last line checks whether the portfolio really is **market neutral** — its correlation with SPY should be near zero.

In [ ]:
port_k = kal.loc[test].mean(axis=1)
port_s = sta.loc[test].mean(axis=1)
spy = close["SPY"].pct_change().loc[test].fillna(0)

def summary(r, label):
    eq = (1 + r).cumprod()
    yrs = len(r) / 252
    cagr = eq.iloc[-1] ** (1 / yrs) - 1
    dd = (eq / eq.cummax() - 1).min()
    print(f"{label:24s} annual {cagr:+.1%}  ·  vol {r.std()*np.sqrt(252):.1%}  ·  "
          f"Sharpe {sharpe(r):+.2f}  ·  worst drawdown {dd:.1%}")
    return eq

eq_k = summary(port_k, "Basket, rolling hedge")
eq_s = summary(port_s, "Basket, fixed hedge")
eq_m = summary(spy,    "Just holding SPY")
print(f"\nCorrelation with SPY — rolling basket: {port_k.corr(spy):+.2f}   ·   fixed basket: {port_s.corr(spy):+.2f}")

single_best = tbl.iloc[0]["Pair"]
print(f"Best single pair Sharpe: {tbl.iloc[0]['Rolling: Sharpe']:+.2f} ({single_best}) — "
      f"but you could only know which pair that was in hindsight.")

fig, ax = plt.subplots(2, 1, figsize=(10, 7))
ax[0].plot(eq_k.index, eq_k, label="Basket, rolling hedge", linewidth=1.6)
ax[0].plot(eq_s.index, eq_s, label="Basket, fixed hedge", linewidth=1.2)
ax[0].axhline(1, color="black", linewidth=0.8); ax[0].set_title("Growth of $1 — judging period"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(eq_m.index, eq_m, color="gray", label="SPY", linewidth=1.2)
ax[1].plot(eq_k.index, eq_k, label="Basket, rolling hedge", linewidth=1.6)
ax[1].axhline(1, color="black", linewidth=0.8); ax[1].set_title("Compared with the market"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## How to read the results

- **Rolling vs. fixed:** if the rolling hedge beats the fixed one across most pairs, adapting to drifting relationships matters.
- **Basket vs. single pair:** the portfolio's Sharpe is usually steadier than most individual pairs — that's diversification.
- **Correlation with SPY near zero:** the strategy's results don't depend on whether the market goes up or down.
  That's the whole appeal — returns that don't move with the rest of your portfolio.
- **Small returns are normal.** Market-neutral strategies typically earn single-digit returns with low volatility.
  Funds lever them up several times, which is how a 3% unlevered return becomes meaningful — and also why they blow up when correlations break.

## Your exercises

1. **Step 3:** change `DELTA` to `1e-6` and `1e-8`, then run all. Watch what happens when it adapts too fast or too slow.
2. **Step 4:** try `ENTRY_Z = 1.0` and `2.0`. Fewer, bigger trades or more, smaller ones?
3. **Step 2:** add pairs from an industry you know — two homebuilders, two banks, two chip makers.
4. **Step 5:** do the pairs that did well here match the ones that passed the cointegration test in Lesson 7?

**For GitHub:** Lessons 7 and 8 together tell a complete research story — a naive approach that broke, a diagnosis,
and a professional fix tested honestly.